# Weighted centroid 
**the “centre of mass” of the ERPAC map**
- Where is the ERPAC activity centred overall?

In [50]:
import os
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from scipy.stats import ttest_ind, t
from mne.stats import permutation_cluster_test

import os
import sys
from pathlib import Path
PROJECT_ROOT = Path.cwd().parent
sys.path.append(str(PROJECT_ROOT))
from config.paths import ERPAC_DIR, ROI_STCS_DIR
from config.config import COUPLINGS, TASK_STAGES, ROI, TOI, GROUPS, SUBJECTS

%matplotlib qt

In [51]:
SUBJECTS

{'Y': ['s1_pac_sub01',
  's1_pac_sub07',
  's1_pac_sub10',
  's1_pac_sub11',
  's1_pac_sub22',
  's1_pac_sub24',
  's1_pac_sub26',
  's1_pac_sub29',
  's1_pac_sub32',
  's1_pac_sub33',
  's1_pac_sub38',
  's1_pac_sub43',
  's1_pac_sub52',
  's1_pac_sub58',
  's1_pac_sub59',
  's1_pac_sub61',
  's1_pac_sub63',
  's1_pac_sub64',
  's1_pac_sub66',
  's1_pac_sub67',
  's1_pac_sub71',
  's1_pac_sub76',
  's1_pac_sub77'],
 'O': ['s1_pac_sub12',
  's1_pac_sub14',
  's1_pac_sub15',
  's1_pac_sub17',
  's1_pac_sub20',
  's1_pac_sub21',
  's1_pac_sub23',
  's1_pac_sub34',
  's1_pac_sub35',
  's1_pac_sub36',
  's1_pac_sub40',
  's1_pac_sub41',
  's1_pac_sub45',
  's1_pac_sub47',
  's1_pac_sub48',
  's1_pac_sub49',
  's1_pac_sub50',
  's1_pac_sub53',
  's1_pac_sub54',
  's1_pac_sub55',
  's1_pac_sub56',
  's1_pac_sub60',
  's1_pac_sub62',
  's1_pac_sub68']}

In [52]:
# ============================================================
# 1. LOAD TIME-RESOLVED ERPAC DATA
# ============================================================

erpac_df = pd.read_parquet(
    os.path.join(
        ERPAC_DIR,
        "erpac_results.parquet"
    )
)

erpac_df_toi = erpac_df[
    (
        (erpac_df["task_stage"] == "plan") &
        erpac_df["time"].between(
            TOI["plan"]["start"],
            TOI["plan"]["end"]
        )
    )
    |
    (
        (erpac_df["task_stage"] == "go") &
        erpac_df["time"].between(
            TOI["go"]["start"],
            TOI["go"]["end"]
        )
    )
].copy()

erpac_df_toi

,sub,group,task,task_stage,coupling,roi,amp_freq,time,erpac_value
176,s1_pac_sub12,O,FTT,go,alpha_gamma,M1,32.5,-0.148,0.129945
177,s1_pac_sub12,O,FTT,go,alpha_gamma,M1,32.5,-0.146,0.131710
178,s1_pac_sub12,O,FTT,go,alpha_gamma,M1,32.5,-0.144,0.133261
179,s1_pac_sub12,O,FTT,go,alpha_gamma,M1,32.5,-0.142,0.134622
180,s1_pac_sub12,O,FTT,go,alpha_gamma,M1,32.5,-0.140,0.136041
...,...,...,...,...,...,...,...,...,...
25430755,s1_pac_sub77,Y,FTT,plan,theta_gamma,SMA,76.5,0.492,0.153508
25430756,s1_pac_sub77,Y,FTT,plan,theta_gamma,SMA,76.5,0.494,0.151223
25430757,s1_pac_sub77,Y,FTT,plan,theta_gamma,SMA,76.5,0.496,0.145920
25430758,s1_pac_sub77,Y,FTT,plan,theta_gamma,SMA,76.5,0.498,0.146189


In [53]:
# Extract one participant's ERPAC map

def get_subject_erpac_map(
    erpac_df,
    sub,
    group,
    coupling,
    task_stage,
    roi,
    task="FTT",
):
    df = erpac_df[
        (erpac_df["sub"] == sub) &
        (erpac_df["group"] == group) &
        (erpac_df["task"] == task) &
        (erpac_df["coupling"] == coupling) &
        (erpac_df["task_stage"] == task_stage) &
        (erpac_df["roi"] == roi)
    ].copy()

    pivot = df.pivot(
        index="amp_freq",
        columns="time",
        values="erpac_value",
    )

    pivot = pivot.sort_index(
        axis=0
    ).sort_index(
        axis=1
    )

    freqs = pivot.index.to_numpy(
        dtype=float
    )

    times = pivot.columns.to_numpy(
        dtype=float
    )

    erpac_map = pivot.to_numpy()

    return erpac_map, freqs, times



# ============================================================
# Weighted centroid
# ===========================================================

def erpac_centroid(
    erpac_map,
    freqs,
    times,
    top_percent=20,
):
    """
    Weighted centroid of strongest ERPAC values.

    Parameters
    ----------
    erpac_map : ndarray
        Shape (n_freqs, n_times).

    freqs : ndarray
        Gamma frequencies.

    times : ndarray
        Time points.

    top_percent : float or None
        Percentage of strongest ERPAC bins to retain.
        Example:
            20 = top 20%
            None = use whole map

    Returns
    -------
    centroid_time
    centroid_freq
    """
    
    x = erpac_map.copy()

    # Remove invalid values
    x[~np.isfinite(x)] = np.nan

    if np.all(np.isnan(x)):
        return np.nan, np.nan

    # ----------------------------------------
    # Restrict to strongest ERPAC values
    # ----------------------------------------

    if top_percent is not None:
        percentile = 100 - top_percent

        threshold = np.nanpercentile(
            x,
            percentile
        )

        weights = np.where(
            x >= threshold,
            x,
            0
        )

    else:
        weights = np.nan_to_num(
            x,
            nan=0
        )

    # Ensure non-negative weights
    weights = np.clip(
        weights,
        0,
        None
    )

    total_weight = np.sum(weights)

    if total_weight == 0:
        return np.nan, np.nan

    # ----------------------------------------
    # Coordinates
    # ----------------------------------------

    freq_grid, time_grid = np.meshgrid(
        freqs,
        times,
        indexing="ij"
    )

    centroid_time = (
        np.sum(weights * time_grid)
        / total_weight
    )

    centroid_freq = (
        np.sum(weights * freq_grid)
        / total_weight
    )

    return centroid_time, centroid_freq

In [54]:
sub = "s1_pac_sub01"
group = "Y"
coupling = "theta_gamma"
task_stage = "plan"
roi = "SMA"

erpac_map, freqs, times = get_subject_erpac_map(
    erpac_df_toi,
    sub=sub,
    group=group,
    coupling=coupling,
    task_stage=task_stage,
    roi=roi,
)

centroid_time, centroid_freq = erpac_centroid(
    erpac_map,
    freqs,
    times,
    top_percent=20,
)

print(
    f"centroid time: {centroid_time:.2f} s\ncentroid frequency: {centroid_freq:.2f} Hz"
)

centroid time: 0.23 s
centroid frequency: 50.74 Hz


In [55]:
def calculate_all_centroids(
    erpac_df,
    coupling,
    task_stage,
    roi,
    task="FTT",
    top_percent=20,
):
    rows = []

    df_condition = erpac_df[
        (erpac_df["task"] == task) &
        (erpac_df["coupling"] == coupling) &
        (erpac_df["task_stage"] == task_stage) &
        (erpac_df["roi"] == roi)
    ]

    for group in GROUPS:

        subs = sorted(
            df_condition[
                df_condition["group"] == group
            ]["sub"].unique()
        )

        for sub in subs:

            erpac_map, freqs, times = get_subject_erpac_map(
                erpac_df=erpac_df,
                sub=sub,
                group=group,
                coupling=coupling,
                task_stage=task_stage,
                roi=roi,
                task=task,
            )

            centroid_time, centroid_freq = erpac_centroid(
                erpac_map=erpac_map,
                freqs=freqs,
                times=times,
                top_percent=top_percent,
            )

            rows.append({
                "sub": sub,
                "group": group,
                "coupling": coupling,
                "task_stage": task_stage,
                "roi": roi,
                "centroid_time": centroid_time,
                "centroid_freq": centroid_freq,
            })

    return pd.DataFrame(rows)

In [56]:
centroid_df = calculate_all_centroids(
    erpac_df_toi,
    coupling=coupling,
    task_stage=task_stage,
    roi=roi,
    top_percent=20,
)


In [57]:
centroid_df

,sub,group,coupling,task_stage,roi,centroid_time,centroid_freq
0,s1_pac_sub01,Y,theta_gamma,plan,SMA,0.230121,50.735763
1,s1_pac_sub07,Y,theta_gamma,plan,SMA,0.270223,58.966059
2,s1_pac_sub10,Y,theta_gamma,plan,SMA,0.249846,53.061807
3,s1_pac_sub11,Y,theta_gamma,plan,SMA,0.267594,52.634122
4,s1_pac_sub22,Y,theta_gamma,plan,SMA,0.248820,52.913415
5,s1_pac_sub24,Y,theta_gamma,plan,SMA,0.221219,48.422502
6,s1_pac_sub26,Y,theta_gamma,plan,SMA,0.215091,52.137924
7,s1_pac_sub29,Y,theta_gamma,plan,SMA,0.258308,53.195741
8,s1_pac_sub32,Y,theta_gamma,plan,SMA,0.199400,55.068959
9,s1_pac_sub33,Y,theta_gamma,plan,SMA,0.260254,57.768729


In [60]:
centroid_df.describe()

,centroid_time,centroid_freq
count,47.000000,47.000000
mean,0.249894,53.841681
std,0.026941,3.139991
min,0.199400,46.497508
25%,0.229150,52.109460
50%,0.249650,53.195741
75%,0.267040,55.897931
max,0.309799,62.614068


In [61]:
centroid_df.groupby(["group", "task_stage", "coupling"]).describe()

centroid_time                                \
                                     count      mean       std       min   
group task_stage coupling                                                  
O     plan       theta_gamma          24.0  0.248148  0.026172  0.199511   
Y     plan       theta_gamma          23.0  0.251716  0.028191  0.199400   

                                                                      \
                                   25%       50%       75%       max   
group task_stage coupling                                              
O     plan       theta_gamma  0.228178  0.242654  0.266406  0.301324   
Y     plan       theta_gamma  0.230610  0.258308  0.268908  0.309799   

                             centroid_freq                                  \
                                     count       mean       std        min   
group task_stage coupling                                                    
O     plan       theta_gamma          24.0  53.732625  3.240046  46.497508   
Y     plan       theta_gamma          23.0  53.955479  3.100617  48.422502   

                                                                          
                                    25%        50%        75%        max  
group task_stage coupling                                                 
O     plan       theta_gamma  51.880045  53.511493  55.839458  60.327403  
Y     plan       theta_gamma  52.162505  52.989517  55.701669  62.614068

ALL CONDITIONS

In [62]:
centroid_dfs = []

for coupling in erpac_df_toi["coupling"].unique():

    for task_stage in erpac_df_toi["task_stage"].unique():

        for roi in erpac_df_toi["roi"].unique():

            print(
                f"Processing: "
                f"{coupling} | {task_stage} | {roi}"
            )

            centroid_df = calculate_all_centroids(
                erpac_df_toi,
                coupling=coupling,
                task_stage=task_stage,
                roi=roi,
                top_percent=20,
            )

            # Add condition identifiers if your function
            # does not already include them
            centroid_df["coupling"] = coupling
            centroid_df["task_stage"] = task_stage
            centroid_df["roi"] = roi

            centroid_dfs.append(
                centroid_df
            )


# ============================================================
# COMBINE EVERYTHING
# ============================================================

centroid_df_all = pd.concat(
    centroid_dfs,
    ignore_index=True
)

print(
    centroid_df_all.shape
)

print(
    centroid_df_all.head()
)

Processing: alpha_gamma | go | M1
Processing: alpha_gamma | go | PMC
Processing: alpha_gamma | go | S1
Processing: alpha_gamma | go | SMA
Processing: alpha_gamma | plan | M1
Processing: alpha_gamma | plan | PMC
Processing: alpha_gamma | plan | S1
Processing: alpha_gamma | plan | SMA
Processing: beta_gamma | go | M1
Processing: beta_gamma | go | PMC
Processing: beta_gamma | go | S1
Processing: beta_gamma | go | SMA
Processing: beta_gamma | plan | M1
Processing: beta_gamma | plan | PMC
Processing: beta_gamma | plan | S1
Processing: beta_gamma | plan | SMA
Processing: theta_gamma | go | M1
Processing: theta_gamma | go | PMC
Processing: theta_gamma | go | S1
Processing: theta_gamma | go | SMA
Processing: theta_gamma | plan | M1
Processing: theta_gamma | plan | PMC
Processing: theta_gamma | plan | S1
Processing: theta_gamma | plan | SMA
(1128, 7)
            sub group     coupling task_stage roi  centroid_time  \
0  s1_pac_sub01     Y  alpha_gamma         go  M1       0.166406   
1  s1_pac_

In [63]:
print(
    centroid_df_all.columns
)

print(
    centroid_df_all.groupby(
        [
            "coupling",
            "task_stage",
            "roi",
        ]
    ).size()
)

Index(['sub', 'group', 'coupling', 'task_stage', 'roi', 'centroid_time',
       'centroid_freq'],
      dtype='str')
coupling     task_stage  roi
alpha_gamma  go          M1     47
                         PMC    47
                         S1     47
                         SMA    47
             plan        M1     47
                         PMC    47
                         S1     47
                         SMA    47
beta_gamma   go          M1     47
                         PMC    47
                         S1     47
                         SMA    47
             plan        M1     47
                         PMC    47
                         S1     47
                         SMA    47
theta_gamma  go          M1     47
                         PMC    47
                         S1     47
                         SMA    47
             plan        M1     47
                         PMC    47
                         S1     47
                         SMA    47
dtype: int64


In [ ]:
centroid_df_all.to_csv(os.path.join(
    ERPAC_DIR,
    "erpac_centroids_all_conditions.csv"),
    index=False
)

In [71]:
centroid_df_all.groupby(
    [   "group",
        "coupling",
        "task_stage",
        "roi",
    ]
).describe(    
    ).to_csv(os.path.join(
    ERPAC_DIR,
    "erpac_centroids_describe.csv")
)

In [70]:
centroid_df_all.groupby(
    [   "group",
        "coupling",
        "task_stage",
        "roi",
    ]
).describe()

centroid_time                                \
                                         count      mean       std       min   
group coupling    task_stage roi                                               
O     alpha_gamma go         M1           24.0  0.170628  0.043097  0.086756   
                             PMC          24.0  0.177874  0.035390  0.116521   
                             S1           24.0  0.169927  0.043488  0.081750   
                             SMA          24.0  0.162354  0.041804  0.096722   
                  plan       M1           24.0  0.236748  0.046041  0.119063   
                             PMC          24.0  0.244678  0.034225  0.169655   
                             S1           24.0  0.246314  0.039025  0.164611   
                             SMA          24.0  0.242971  0.042009  0.144415   
      beta_gamma  go         M1           24.0  0.169482  0.029238  0.099810   
                             PMC          24.0  0.158363  0.037738  0.073251   
                             S1           24.0  0.172190  0.033132  0.081410   
                             SMA          24.0  0.177427  0.029556  0.126261   
                  plan       M1           24.0  0.256327  0.022964  0.211429   
                             PMC          24.0  0.252701  0.025275  0.212344   
                             S1           24.0  0.239408  0.031722  0.160351   
                             SMA          24.0  0.247437  0.029176  0.210560   
      theta_gamma go         M1           24.0  0.164389  0.039581  0.081577   
                             PMC          24.0  0.162121  0.041548  0.084019   
                             S1           24.0  0.161880  0.027540  0.092567   
                             SMA          24.0  0.173043  0.026166  0.116120   
                  plan       M1           24.0  0.253738  0.031164  0.202954   
                             PMC          24.0  0.246562  0.035948  0.184034   
                             S1           24.0  0.251320  0.027590  0.213678   
                             SMA          24.0  0.248148  0.026172  0.199511   
Y     alpha_gamma go         M1           23.0  0.184337  0.040268  0.095041   
                             PMC          23.0  0.186562  0.042232  0.109869   
                             S1           23.0  0.176698  0.037521  0.071123   
                             SMA          23.0  0.179671  0.041998  0.100188   
                  plan       M1           23.0  0.241666  0.039791  0.172915   
                             PMC          23.0  0.242840  0.029503  0.195052   
                             S1           23.0  0.245329  0.039443  0.174695   
                             SMA          23.0  0.239027  0.026397  0.178922   
      beta_gamma  go         M1           23.0  0.176459  0.028718  0.124274   
                             PMC          23.0  0.170801  0.035216  0.087293   
                             S1           23.0  0.167131  0.023827  0.123541   
                             SMA          23.0  0.177573  0.029958  0.117128   
                  plan       M1           23.0  0.250970  0.027909  0.211713   
                             PMC          23.0  0.253778  0.035409  0.172190   
                             S1           23.0  0.251010  0.028553  0.194221   
                             SMA          23.0  0.257715  0.030817  0.205714   
      theta_gamma go         M1           23.0  0.162436  0.034843  0.085850   
                             PMC          23.0  0.173300  0.039090  0.110418   
                             S1           23.0  0.162302  0.037135  0.111553   
                             SMA          23.0  0.170364  0.033496  0.095168   
                  plan       M1           23.0  0.250900  0.031303  0.192204   
                             PMC          23.0  0.249651  0.024025  0.208771   
                             S1           23.0  0.249076  0.041776  0.176169   
                             SMA 

VIZ

In [86]:
def plot_centroid_scatter(
    centroid_df,
    coupling,
    roi,
):
    df = centroid_df[
        (centroid_df["coupling"] == coupling)
        & (centroid_df["roi"] == roi)
    ].copy()

    fig, ax = plt.subplots(figsize=(7, 5))

    markers = {
        "plan": "o",
        "go": "^",
    }

    for (group, stage), d in df.groupby(
        ["group", "task_stage"]
    ):
        ax.scatter(
            d["centroid_time"],
            d["centroid_freq"],
            marker=markers.get(stage, "o"),
            label=f"{group} – {stage}",
            alpha=0.7,
        )

    ax.set_xlabel("Centroid time (s)")
    ax.set_ylabel("Centroid amplitude frequency (Hz)")

    ax.set_title(
        f"{coupling} | {roi}"
    )
    ax.set_xlim(0, 0.4)
    ax.set_ylim(45, 65)

    ax.legend()
    ax.axvline(0, linewidth=1)

    fig.tight_layout()

    return fig

In [90]:
def plot_centroid_distribution(
    centroid_df,
    variable,
    coupling,
    roi,
):
    df = centroid_df[
        (centroid_df["coupling"] == coupling)
        & (centroid_df["roi"] == roi)
    ].copy()

    conditions = (
        df[
            ["group", "task_stage"]
        ]
        .drop_duplicates()
        .reset_index(drop=True)
    )

    fig, ax = plt.subplots(figsize=(7, 5))

    data = []
    labels = []

    for _, row in conditions.iterrows():

        values = df.loc[
            (df["group"] == row["group"])
            & (
                df["task_stage"]
                == row["task_stage"]
            ),
            variable,
        ].dropna()

        data.append(values)

        labels.append(
            f"{row['group']}\n"
            f"{row['task_stage']}"
        )

    ax.boxplot(
        data,
        tick_labels=labels,
        showfliers=False,
    )

    # individual observations
    for i, values in enumerate(
        data,
        start=1,
    ):
        ax.scatter(
            [i] * len(values),
            values,
            alpha=0.5,
        )

    ylabel = {
        "centroid_time":
            "Centroid time (s)",
        "centroid_freq":
            "Centroid amplitude frequency (Hz)",
    }.get(
        variable,
        variable,
    )

    ax.set_ylabel(ylabel)
    if variable == "centroid_time":
        ax.set_ylim(0, 0.38)
    else:
        ax.set_ylim(45, 65)

    ax.set_title(
        f"{coupling} | {roi}"
    )

    fig.tight_layout()

    return fig

In [ ]:
centroids_save_dir = os.path.join(
    ERPAC_DIR,
    "sample_characteristics",
    "centroids"
)

for coupling in COUPLINGS:

    for roi in ROI:
        
        fig1 = plot_centroid_scatter(
            centroid_df_all,
            coupling=coupling,
            roi=roi
        )

        fig2 = plot_centroid_distribution(
            centroid_df_all,
            variable="centroid_freq", # "centroid_time" or "centroid_freq"
            coupling=coupling,
            roi=roi
        )
        
        fig3 = plot_centroid_distribution(
            centroid_df_all,
            variable="centroid_time", # "centroid_time" or "centroid_freq"
            coupling=coupling,
            roi=roi
        )

        fig1.savefig(
            os.path.join(centroids_save_dir, f"centroid_scatter_{coupling}_{roi}.png"),
            dpi=300
        )

        fig2.savefig(
            os.path.join(centroids_save_dir, f"centroid_freq_{coupling}_{roi}.png"),
            dpi=300
        )

        fig3.savefig(
            os.path.join(centroids_save_dir, f"centroid_time_{coupling}_{roi}.png"),
            dpi=300
        )



C:\Users\a1902989\AppData\Local\Temp\ipykernel_26880\3351223820.py:20: RuntimeWarning: More than 20 figures have been opened. Figures created through the pyplot interface (`matplotlib.pyplot.figure`) are retained until explicitly closed and may consume too much memory. (To control this warning, see the rcParam `figure.max_open_warning`). Consider using `matplotlib.pyplot.close()`.
  fig, ax = plt.subplots(figsize=(7, 5))


In [83]:
print(centroids_save_dir)

F:\# study 2\eeg_data\erpac\sample_characteristics\centroids


In [ ]:
centroid_summary = (
    centroid_df_all
    .groupby(
        [
            "group",
            "coupling",
            "task_stage",
            "roi",
        ]
    )
    .agg(
        n=(
            "sub",
            "nunique",
        ),

        time_mean=(
            "centroid_time",
            "mean",
        ),
        time_sd=(
            "centroid_time",
            "std",
        ),

        freq_mean=(
            "centroid_freq",
            "mean",
        ),
        freq_sd=(
            "centroid_freq",
            "std",
        ),
    )
    .reset_index()
)